# Object Detection with YOLOS Small
This notebook uses the `hustvl/yolos-small` model from Hugging Face to perform object detection on images in the `images` folder.

In [ ]:
import os
import torch
from transformers import YolosImageProcessor, YolosForObjectDetection
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

# --- FORCE STABILITY: Deep Device Check ---
def get_best_stable_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        dev = torch.device('cuda')
        dummy_input = torch.randn(1, 3, 32, 32).to(dev)
        dummy_model = torch.nn.Conv2d(3, 3, 3).to(dev)
        with torch.no_grad():
            dummy_model(dummy_input)
        print(f"CUDA is stable. Using: {torch.cuda.get_device_name(0)}")
        return 'cuda'
    except Exception as e:
        print(f"CUDA detected but incompatible ({e}). Falling back to CPU.")
        return 'cpu'

device = get_best_stable_device()

## Load Model and Image Processor

In [ ]:
model_name = "hustvl/yolos-small"
print(f"Loading model {model_name}...")

image_processor = YolosImageProcessor.from_pretrained(model_name)
model = YolosForObjectDetection.from_pretrained(model_name).to(device)

print(f"Ready on {device}.")

## Detection Function

In [ ]:
def detect_objects(image_path, threshold=0.7):
    image = Image.open(image_path).convert("RGB")
    inputs = image_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Post-process results
    target_sizes = torch.tensor([image.size[::-1]]).to(device)
    results = image_processor.post_process_object_detection(outputs, threshold=threshold, target_sizes=target_sizes)[0]

    # Draw boxes
    draw = ImageDraw.Draw(image)
    
    # Try to load a font, fallback to default
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 20)
    except:
        font = ImageFont.load_default()
    
    if results["labels"].numel() > 0:
        for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
            box = [round(i, 2) for i in box.tolist()]
            label_name = model.config.id2label[label.item()]
            
            # Draw rectangle
            draw.rectangle(box, outline="red", width=3)
            
            # Draw label
            text = f"{label_name} {score:.2f}"
            draw.text((box[0], box[1] - 25), text, fill="red", font=font)

    return image, results

## Run Detection

In [ ]:
image_folder = "../images"
image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

if not image_files:
    print("No images found.")
else:
    for img_file in image_files:
        img_path = os.path.join(image_folder, img_file)
        print(f"\nDetecting in: {img_file}")
        
        try:
            processed_img, detections = detect_objects(img_path)
            
            plt.figure(figsize=(10, 6))
            plt.imshow(processed_img)
            plt.title(f"Detections in {img_file}")
            plt.axis('off')
            plt.show()
            
            num_objects = len(detections["labels"])
            print(f"Found {num_objects} objects.")
        except Exception as e:
            print(f"Error processing {img_file}: {e}")